# 1 · Data audit

Before modelling anything, check what the data actually is. This notebook validates the
structure the feature pipeline relies on, then looks for traces of how the data was generated.
The answer shapes every later decision: **the dataset is synthetic**, so it is a sandbox for
methodology, not a source of football insight.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, cross_val_predict

from player_rating.config import DATA_PATH, DATA_SHA256
from player_rating.data import file_sha256, load_raw, validate

raw = load_raw(DATA_PATH)
validate(raw)  # raises if a structural assumption is broken
print("checksum matches the committed results:", file_sha256(DATA_PATH) == DATA_SHA256)
pd.Series(
    {
        "rows": len(raw),
        "columns": raw.shape[1],
        "players": raw["player_id"].nunique(),
        "teams": raw["team"].nunique(),
        "matches": raw["match_id"].nunique(),
        "first date": raw["match_date"].min().date(),
        "last date": raw["match_date"].max().date(),
    }
).to_frame("value")

checksum matches the committed results: True


,value
rows,54600
columns,75
players,1248
teams,48
matches,1050
first date,2026-06-11
last date,2026-07-31


## Structure

`validate()` passed, so:

- there are no missing values;
- every match has exactly two teams with 26-player squads;
- a player appears at most once per match;
- the static attributes (age, height, position, market value, club, ...) never change between
  matches, so they cannot carry information from later matches.

## Is this a real tournament?

A real 48-team World Cup has 104 matches, and no team plays more than 8 of them.

In [2]:
matches_per_team = raw.groupby("team")["match_id"].nunique()
matches_per_team_day = raw.groupby(["team", "match_date"])["match_id"].nunique()
pd.Series(
    {
        "matches per team (min)": matches_per_team.min(),
        "matches per team (max)": matches_per_team.max(),
        "cases of a team playing twice on the same day": int((matches_per_team_day > 1).sum()),
    }
).to_frame("value")

,value
matches per team (min),33
matches per team (max),66
cases of a team playing twice on the same day,521


The tournament stages are not ordered in time either. Matches labelled "Final" appear in seven
of the eight weeks, mixed in with group matches:

In [3]:
week = raw["match_date"].dt.to_period("W-SUN").dt.start_time.dt.date.rename("week starting")
stage_share = pd.crosstab(week, raw["tournament_stage"], normalize="index")
stage_share[["Group Stage", "Round of 32", "Quarter Finals", "Final"]].round(3)

tournament_stage,Group Stage,Round of 32,Quarter Finals,Final
week starting,,,,
2026-06-08,0.485,0.250,0.103,0.000
2026-06-15,0.549,0.170,0.111,0.026
2026-06-22,0.593,0.153,0.093,0.008
2026-06-29,0.546,0.153,0.067,0.006
2026-07-06,0.558,0.138,0.065,0.043
2026-07-13,0.559,0.162,0.088,0.029
2026-07-20,0.529,0.190,0.059,0.026
2026-07-27,0.562,0.182,0.066,0.008


Player names are recombinations of real players' first and last names:

In [4]:
raw.drop_duplicates("player_id").query("team == 'Spain'")["player_name"].head(8).tolist()

['Rodri Fati',
 'Ansu Le Normand',
 'Gavi Ramos',
 'Pedro Cubarsi',
 'Alvaro Oyarzabal',
 'Dani Gavi',
 'Rodri Laporte',
 'Ansu Fati']

## What drives the rating?

Real rating systems compute a player's rating from what they did in the match. That can be
tested here: predict `player_rating` from the events recorded in the same match, and then from
the dataset's composite scores (`performance_score`, `creativity_score`, ...).

Both use information from *during* the match, so this is an audit only. None of these columns
is ever a model feature.

In [5]:
played = raw[raw["minutes_played"] >= 30]
events = [
    "minutes_played",
    "goals",
    "assists",
    "shots",
    "shots_on_target",
    "expected_goals_xg",
    "expected_assists_xa",
    "key_passes",
    "successful_passes",
    "total_passes",
    "pass_accuracy",
    "dribbles_attempted",
    "successful_dribbles",
    "crosses",
    "successful_crosses",
    "tackles",
    "interceptions",
    "clearances",
    "blocks",
    "aerial_duels_won",
    "aerial_duels_lost",
    "recoveries",
    "defensive_actions",
    "fouls_committed",
    "fouls_suffered",
    "yellow_cards",
    "red_cards",
    "offsides",
    "saves",
    "save_percentage",
    "punches",
    "clean_sheet",
    "goals_conceded",
    "penalty_saves",
    "distance_covered_km",
    "sprint_distance_km",
    "top_speed_kmh",
    "accelerations",
    "decelerations",
    "stamina_score",
    "goals_team",
    "goals_opponent",
]
composites = [
    "performance_score",
    "offensive_contribution",
    "defensive_contribution",
    "possession_impact",
    "pressure_resistance",
    "creativity_score",
    "consistency_score",
    "clutch_performance_score",
]
folds = KFold(5, shuffle=True, random_state=0)


def explained_share(columns):
    model = HistGradientBoostingRegressor(random_state=0)
    predicted = cross_val_predict(model, played[columns], played["player_rating"], cv=folds)
    return r2_score(played["player_rating"], predicted)


pd.Series(
    {
        f"recorded match events ({len(events)} columns)": explained_share(events),
        f"synthetic composite scores ({len(composites)} columns)": explained_share(composites),
    }
).round(3).to_frame("cross-validated R² for player_rating")

,cross-validated R² for player_rating
recorded match events (42 columns),0.224
synthetic composite scores (8 columns),0.864


The rating is almost a function of the synthetic composite scores, and only loosely related
to goals, passes and tackles. A player's market value, however, is closely tied to their
average rating, which makes it the strongest pre-match feature later on:

In [6]:
per_player = played.groupby("player_id")[["market_value_eur", "player_rating"]].mean()
correlation = np.corrcoef(np.log(per_player["market_value_eur"]), per_player["player_rating"])
print(f"correlation of log market value with a player's average rating: {correlation[0, 1]:.2f}")

correlation of log market value with a player's average rating: 0.79


## Conclusions

- The structure is clean and consistent, so the assumptions of the feature pipeline hold.
- The data is synthetic. The schedules are impossible, the stages are unordered, the names are
  recombined, and the rating is generated from latent scores rather than from match events.
- Consequences for modelling:
  1. the results say nothing about real football;
  2. the predictable share of the rating is set by the generator: a player-level component,
     tied to market value, plus match noise. `02_modeling.ipynb` measures that ceiling;
  3. the dataset is still a good sandbox for the parts of ML work that transfer: leakage-safe
     features, honest validation, model comparison with uncertainty, and calibrated intervals.